In [9]:
import cv2
import numpy as np

# ═════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ═════════════════════════════════════════════════════════════════════════════
SYRINGE_ML   = 10.0
SCALE        = 0.33    # 1/3 resolution → ~10ms/frame, well under 1s latency
MARK_DARK    = 85
PLUNGER_DARK = 65
# ═════════════════════════════════════════════════════════════════════════════

FONT      = cv2.FONT_HERSHEY_SIMPLEX
FONT_BOLD = cv2.FONT_HERSHEY_DUPLEX
url = "https://192.168.1.2:8080/video" 



In [10]:
def detect(gray):
    h, w   = gray.shape
    small  = cv2.resize(gray, (int(w * SCALE), int(h * SCALE)))
    sh, sw = small.shape
    dark   = (small < MARK_DARK).astype(np.int8)

    # Best scan column: most consistent evenly-spaced thin dark marks
    y0s, y1s = int(sh * 0.20), int(sh * 0.80)
    best_x_s, best_marks_s = 0, []
    for x in range(20, sw - 10):
        col    = dark[y0s:y1s, x]
        tr     = np.diff(col)
        starts = np.where(tr == 1)[0]; ends = np.where(tr == -1)[0]
        if len(starts) == 0 or len(ends) == 0: continue
        if ends[0] < starts[0]: ends = ends[1:]
        n = min(len(starts), len(ends)); starts, ends = starts[:n], ends[:n]
        thin    = (ends - starts >= 1) & (ends - starts <= 5)
        centers = ((starts + ends) // 2 + y0s)[thin]
        if len(centers) < 5: continue
        diffs = np.diff(centers)
        for base in range(13, 24):
            ok    = (diffs >= base * 0.75) & (diffs <= base * 1.35)
            group = [centers[0]]
            for i, flag in enumerate(ok):
                if flag: group.append(centers[i + 1])
            if len(group) > len(best_marks_s):
                best_marks_s = group[:]; best_x_s = x

    if len(best_marks_s) < 5:
        return [], 0, 0, None

    best_marks_s = [int(m) for m in best_marks_s[:11]]

    # Barrel walls (local Sobel)
    sobelx     = cv2.Sobel(small, cv2.CV_32F, 1, 0, ksize=3)
    lo = max(0, best_x_s-70); hi = min(sw, best_x_s+70)
    ce = np.abs(sobelx[int(sh*0.25):int(sh*0.75), lo:hi]).sum(axis=0)
    cs = np.convolve(ce, np.ones(3)/3, mode='same')
    left_s  = int(np.argmax(cs[:len(cs)//2])) + lo
    tmp     = cs.copy(); tmp[:len(cs)//2] = 0
    right_s = int(np.argmax(tmp)) + lo
    if right_s <= left_s: right_s = left_s + 30

    # Barrel top — reject any marks above this line
    barrel_top_s = int(sh * 0.12)
    for y in range(int(sh * 0.06), int(sh * 0.55)):
        row = small[y, left_s:right_s]
        if len(row) > 0 and (row < 90).sum() > len(row) * 0.25:
            barrel_top_s = y; break

    best_marks_s = [m for m in best_marks_s if m >= barrel_top_s][:11]
    if len(best_marks_s) < 5:
        return [], 0, 0, None

    # Plunger
    blurred    = cv2.GaussianBlur(small, (5, 5), 0)
    _, thresh  = cv2.threshold(blurred, PLUNGER_DARK, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    plunger_s  = None
    for c in sorted(contours, key=cv2.contourArea, reverse=True):
        x, y, cw, ch = cv2.boundingRect(c)
        if (cv2.contourArea(c) > 300 and y > sh * 0.45
                and x < right_s and (x + cw) > left_s and cw > 10):
            plunger_s = (x, y, cw, ch); break
    
    def up(v): return int(v / SCALE)
    marks   = [up(m) for m in best_marks_s]
    bx1, bx2 = up(left_s), up(right_s)
    plunger = tuple(up(v) for v in plunger_s) if plunger_s else None
    return marks, bx1, bx2, plunger



In [ ]:
def draw(frame, marks, bx1, bx2, plunger, ml):
    out = frame.copy()
    spacing, zero_y = float(np.mean(np.diff(marks))), marks[0]
    plunger_top_y   = plunger[1]
    px, py, pw, ph  = plunger
    label_x         = bx2 + 14

    ov = out.copy()
    cv2.rectangle(ov, (bx1+2, zero_y), (bx2-2, plunger_top_y), (80,160,255), -1)
    cv2.addWeighted(ov, 0.18, out, 0.82, 0, out)

    for i, my in enumerate(marks):
        n = i
        if   n == 0:  c,t,tk,fs = (0,255,100),3,55,0.75
        elif n == 10: c,t,tk,fs = (0,220,255),3,55,0.75
        elif n == 5:  c,t,tk,fs = (255,180, 0),3,50,0.72
        else:         c,t,tk,fs = (255,255, 80),2,35,0.60
        cv2.line(out,(bx1-tk,my),(bx1-2,my),c,t)
        cv2.line(out,(bx1-2,my),(bx2+2,my),c,1)
        cv2.putText(out,f"{n} ml",(label_x,my+7),FONT,fs,c,t)

    cv2.rectangle(out,(px,py),(px+pw,py+ph),(0,230,60),3)
    cv2.line(out,(bx1-5,plunger_top_y),(bx2+5,plunger_top_y),(0,0,255),4)
    cv2.arrowedLine(out,(bx2+130,plunger_top_y),(bx2+12,plunger_top_y),(0,0,255),3,tipLength=0.25)
    cv2.putText(out,"Plunger top",(bx2+140,plunger_top_y-8),FONT,0.55,(0,0,255),2)

    bx,by,bw,bh = 20,20,240,110
    cv2.rectangle(out,(bx,by),(bx+bw,by+bh),(15,15,15),-1)
    cv2.rectangle(out,(bx,by),(bx+bw,by+bh),(0,220,220),3)
    cv2.putText(out,f"{ml} ml",(bx+14,by+68),FONT_BOLD,2.2,(0,255,255),5)
    cv2.putText(out,"PLUNGER ESTIMATE",(bx+14,by+94),FONT,0.45,(160,160,160),1)

    bx2b,by2,bh2 = 20,140,18; bar_max=240
    bar_fill=int((ml/SYRINGE_ML)*bar_max)
    cv2.rectangle(out,(bx2b,by2),(bx2b+bar_max,by2+bh2),(50,50,50),-1)
    cv2.rectangle(out,(bx2b,by2),(bx2b+bar_fill,by2+bh2),(0,200,200),-1)
    cv2.putText(out,f"{ml/SYRINGE_ML*100:.0f}% of {SYRINGE_ML:.0f} ml",
                (bx2b,by2+bh2+22),FONT,0.55,(160,160,160),1)
    return out



In [ ]:
#main loop

cap = cv2.VideoCapture(url)
if not cap.isOpened():
    print("[ERROR] Cannot open camera."); exit()

print("[INFO] Camera opened. Press Q to quit.")
cv2.namedWindow("Syringe Live", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Syringe Live", 800, 600)
last_ml = None

while True:
    ret, frame = cap.read()
    if not ret: break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    marks, bx1, bx2, plunger = detect(gray)

    if len(marks) < 5 or plunger is None:
        # No syringe — plain camera, no overlay
        cv2.imshow("Syringe Live", frame)
        last_ml = None
    else:
        spacing = float(np.mean(np.diff(marks)))
        raw_ml  = (plunger[1] - marks[0]) / spacing
        ml      = round(min(SYRINGE_ML, max(0.0, raw_ml)), 1)  # strict 0-10

        result  = draw(frame, marks, bx1, bx2, plunger, ml)
        cv2.imshow("Syringe Live", result)

        if ml != last_ml:
            print(f"  ► {ml} ml ({ml/SYRINGE_ML*100:.0f}%) | marks:{len(marks)} | plunger_y:{plunger[1]}")
            last_ml = ml

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("[INFO] Done.")

In [5]:
#part 2

In [1]:
import cv2
import numpy as np
from collections import deque

# ═════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ═════════════════════════════════════════════════════════════════════════════
SYRINGE_ML   = 10.0
SCALE        = 0.33
MARK_DARK    = 85
PLUNGER_DARK = 75
SMOOTH_N     = 5     # frames averaged for stable ml readout
# ═════════════════════════════════════════════════════════════════════════════

FONT      = cv2.FONT_HERSHEY_SIMPLEX
FONT_BOLD = cv2.FONT_HERSHEY_DUPLEX

url = "https://192.168.1.2:8080/video" 



In [2]:
def detect(gray):
    h, w   = gray.shape
    small  = cv2.resize(gray, (int(w * SCALE), int(h * SCALE)))
    sh, sw = small.shape
    dark   = (small < MARK_DARK).astype(np.int8)

    # Graduation marks
    y0s, y1s = int(sh * 0.20), int(sh * 0.80)
    best_x_s, best_marks_s = 0, []
    for x in range(20, sw - 10):
        col    = dark[y0s:y1s, x]
        tr     = np.diff(col)
        starts = np.where(tr == 1)[0]; ends = np.where(tr == -1)[0]
        if len(starts) == 0 or len(ends) == 0: continue
        if ends[0] < starts[0]: ends = ends[1:]
        n = min(len(starts), len(ends)); starts, ends = starts[:n], ends[:n]
        thin    = (ends - starts >= 1) & (ends - starts <= 5)
        centers = ((starts + ends) // 2 + y0s)[thin]
        if len(centers) < 5: continue
        diffs = np.diff(centers)
        for base in range(13, 24):
            ok    = (diffs >= base * 0.75) & (diffs <= base * 1.35)
            group = [centers[0]]
            for i, flag in enumerate(ok):
                if flag:
                    group.append(centers[i + 1])
                else:
                    if len(group) > len(best_marks_s): best_marks_s = group[:]; best_x_s = x
                    group = [centers[i + 1]]
            if len(group) > len(best_marks_s): best_marks_s = group[:]; best_x_s = x

    if len(best_marks_s) < 5: return [], 0, 0, None, None
    best_marks_s = [int(m) for m in best_marks_s[:11]]

    # Barrel walls
    sobelx     = cv2.Sobel(small, cv2.CV_32F, 1, 0, ksize=3)
    lo = max(0, best_x_s-70); hi = min(sw, best_x_s+70)
    ce = np.abs(sobelx[int(sh*0.25):int(sh*0.75), lo:hi]).sum(axis=0)
    cs = np.convolve(ce, np.ones(3)/3, mode='same')
    left_s  = int(np.argmax(cs[:len(cs)//2])) + lo
    tmp     = cs.copy(); tmp[:len(cs)//2] = 0
    right_s = int(np.argmax(tmp)) + lo
    if right_s <= left_s: right_s = left_s + 30

    # Barrel top
    barrel_top_s = int(sh * 0.12)
    for y in range(int(sh*0.06), int(sh*0.55)):
        row = small[y, left_s:right_s]
        if len(row) > 0 and (row < 90).sum() > len(row) * 0.25:
            barrel_top_s = y; break

    best_marks_s = [m for m in best_marks_s if m >= barrel_top_s][:11]
    if len(best_marks_s) < 5: return [], 0, 0, None, None
    spacing_s   = float(np.mean(np.diff(best_marks_s)))
    last_mark_s = max(best_marks_s)

    # Robust plunger: row scan bounded to barrel
    scan_end = min(int(last_mark_s + spacing_s * 1.5), sh - 1)
    dense    = []
    for y in range(barrel_top_s, scan_end):
        row = small[y, left_s+2:right_s-2]
        if len(row) == 0: continue
        pct = (row < PLUNGER_DARK).sum() / len(row) * 100
        if pct > 40: dense.append((y, pct))

    groups = []
    if dense:
        cur = [dense[0]]
        for i in range(1, len(dense)):
            if dense[i][0] - dense[i-1][0] <= 4: cur.append(dense[i])
            else:
                if len(cur) >= 3: groups.append(cur)
                cur = [dense[i]]
        if len(cur) >= 3: groups.append(cur)

    if not groups: return [], 0, 0, None, None
    best_g        = max(groups, key=lambda g: len(g) * np.mean([p for _, p in g]))
    plunger_top_s = best_g[0][0]
    plunger_bot_s = best_g[-1][0]

    raw_ml = (plunger_top_s - best_marks_s[0]) / spacing_s
    ml     = round(min(SYRINGE_ML, max(0.0, raw_ml)), 1)

    def up(v): return int(v / SCALE)
    marks   = [up(m) for m in best_marks_s]
    bx1, bx2 = up(left_s), up(right_s)
    plunger = (bx1, up(plunger_top_s), bx2 - bx1, up(plunger_bot_s - plunger_top_s))
    return marks, bx1, bx2, plunger, ml




In [3]:
def draw(frame, marks, bx1, bx2, plunger, ml):
    out = frame.copy()
    spacing, zero_y = float(np.mean(np.diff(marks))), marks[0]
    px, py, pw, ph  = plunger
    label_x         = bx2 + 14

    ov = out.copy()
    cv2.rectangle(ov, (bx1+2, zero_y), (bx2-2, py), (80,160,255), -1)
    cv2.addWeighted(ov, 0.18, out, 0.82, 0, out)

    for i, my in enumerate(marks):
        n = i
        if   n == 0:  c,t,tk,fs = (0,255,100),3,55,0.75
        elif n == 10: c,t,tk,fs = (0,220,255),3,55,0.75
        elif n == 5:  c,t,tk,fs = (255,180, 0),3,50,0.72
        else:         c,t,tk,fs = (255,255, 80),2,35,0.60
        cv2.line(out,(bx1-tk,my),(bx1-2,my),c,t)
        cv2.line(out,(bx1-2,my),(bx2+2,my),c,1)
        cv2.putText(out,f"{n} ml",(label_x,my+7),FONT,fs,c,t)

    cv2.rectangle(out,(px,py),(px+pw,py+ph),(0,230,60),3)
    cv2.line(out,(bx1-5,py),(bx2+5,py),(0,0,255),4)
    cv2.arrowedLine(out,(bx2+130,py),(bx2+12,py),(0,0,255),3,tipLength=0.25)
    cv2.putText(out,"Plunger top",(bx2+140,py-8),FONT,0.55,(0,0,255),2)

    bx,by,bw,bh=20,20,240,110
    cv2.rectangle(out,(bx,by),(bx+bw,by+bh),(15,15,15),-1)
    cv2.rectangle(out,(bx,by),(bx+bw,by+bh),(0,220,220),3)
    cv2.putText(out,f"{ml} ml",(bx+14,by+68),FONT_BOLD,2.2,(0,255,255),5)
    cv2.putText(out,"PLUNGER ESTIMATE",(bx+14,by+94),FONT,0.45,(160,160,160),1)

    bx2b,by2,bh2=20,140,18; bar_max=240
    cv2.rectangle(out,(bx2b,by2),(bx2b+bar_max,by2+bh2),(50,50,50),-1)
    cv2.rectangle(out,(bx2b,by2),(bx2b+int((ml/SYRINGE_ML)*bar_max),by2+bh2),(0,200,200),-1)
    cv2.putText(out,f"{ml/SYRINGE_ML*100:.0f}% of {SYRINGE_ML:.0f} ml",
                (bx2b,by2+bh2+22),FONT,0.55,(160,160,160),1)
    return out




In [4]:
# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(url)
if not cap.isOpened():
    print("[ERROR] Cannot open camera."); exit()

print("[INFO] Camera opened. Press Q to quit.")
cv2.namedWindow("Syringe Live", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Syringe Live", 800, 600)

ml_history = deque(maxlen=SMOOTH_N)
last_print  = None

while True:
    ret, frame = cap.read()
    if not ret: break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    marks, bx1, bx2, plunger, ml_raw = detect(gray)

    if not marks or plunger is None:
        ml_history.clear()
        cv2.imshow("Syringe Live", frame)   # plain camera — no overlay
        last_print = None
    else:
        ml_history.append(ml_raw)
        # Smooth over last N frames, snap to 0.5 ml steps
        ml_smooth = round(round(np.mean(ml_history) * 2) / 2, 1)
        ml_smooth = round(min(SYRINGE_ML, max(0.0, ml_smooth)), 1)

        result = draw(frame, marks, bx1, bx2, plunger, ml_smooth)
        cv2.imshow("Syringe Live", result)

        if ml_smooth != last_print:
            print(f"  ► {ml_smooth} ml ({ml_smooth/SYRINGE_ML*100:.0f}%) | marks:{len(marks)} | plunger_top:{plunger[1]}")
            last_print = ml_smooth

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("[INFO] Done.")

[INFO] Camera opened. Press Q to quit.
  ► 4.0 ml (40%) | marks:5 | plunger_top:827
  ► 0.0 ml (0%) | marks:6 | plunger_top:348
  ► 5.0 ml (50%) | marks:5 | plunger_top:618
  ► 4.5 ml (45%) | marks:5 | plunger_top:424
  ► 6.0 ml (60%) | marks:8 | plunger_top:769
  ► 9.5 ml (95%) | marks:10 | plunger_top:857
  ► 9.5 ml (95%) | marks:10 | plunger_top:796
  ► 8.5 ml (85%) | marks:9 | plunger_top:793
  ► 7.0 ml (70%) | marks:5 | plunger_top:551
  ► 9.5 ml (95%) | marks:11 | plunger_top:806
  ► 9.5 ml (95%) | marks:10 | plunger_top:824
  ► 9.5 ml (95%) | marks:10 | plunger_top:781
  ► 4.0 ml (40%) | marks:5 | plunger_top:530
  ► 6.0 ml (60%) | marks:9 | plunger_top:754
  ► 7.5 ml (75%) | marks:10 | plunger_top:751
  ► 8.0 ml (80%) | marks:10 | plunger_top:751
  ► 9.5 ml (95%) | marks:10 | plunger_top:748
  ► 8.5 ml (85%) | marks:9 | plunger_top:754
  ► 9.0 ml (90%) | marks:11 | plunger_top:757
  ► 9.5 ml (95%) | marks:11 | plunger_top:757
  ► 9.5 ml (95%) | marks:10 | plunger_top:724
  ► 9.